# WeatherDataWarehouse in Google Colab

Runs the full pipeline: load daily weather for 20 Indian cities from the Open-Meteo Archive API into DuckDB, build the dbt models, run the data tests, and query the marts.

Runtime > Run all. No API key is needed.

In [ ]:
!git clone https://github.com/rakshit-bahadur/weather-data-warehouse.git
%cd weather-data-warehouse
!pip install -q -r requirements.txt

## 1. Load raw data
Loads the last 90 days for every city in `config/cities.yml`. Re-running is safe: each city's date range is replaced, not appended.

In [ ]:
!python -m ingestion.fetch_weather --days 90

To run without internet access, load the bundled synthetic sample instead:
```
!python -m ingestion.fetch_weather --sample-dir data/sample
```

In [ ]:
!python -m pytest -q

## 2. Build and test the warehouse with dbt

In [ ]:
%cd dbt
!dbt build --profiles-dir .
%cd ..

## 3. Explore the marts

In [ ]:
import duckdb
con = duckdb.connect('warehouse/weather.duckdb', read_only=True)
con.sql("""
    select city, region, round(avg(avg_temp_c), 1) as avg_temp_c, sum(total_rain_mm) as rain_mm
    from marts.mart_city_monthly
    group by city, region
    order by avg_temp_c desc
""").df()

In [ ]:
con.sql("""
    select city, weather_date, temp_mean_c, baseline_mean_c, z_score, anomaly_type
    from marts.mart_temperature_anomalies
    order by abs(z_score) desc
    limit 15
""").df()

In [ ]:
trend = con.sql("""
    select weather_date, city, temp_mean_7d_avg_c
    from marts.fct_daily_weather
    where city in ('Delhi', 'Mumbai', 'Bengaluru', 'Patna')
""").df()
trend.pivot(index='weather_date', columns='city', values='temp_mean_7d_avg_c').plot(figsize=(11, 4), title='7-day average temperature (°C)');

In [ ]:
con.sql('select * from raw.load_audit order by loaded_at desc limit 20').df()